[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module0_TimeSeries/09_MLPForecasting.ipynb#copy=true)

# Multilayer Perceptrons for Time-Series Forecasting

**Module 0 · Lesson 9 of 13 · Student edition**  
**Estimated class time:** 80–95 minutes  
**Source sequence:** Original Day 3  

**Prerequisite:** Lessons 7–8  

## Learning objectives

By the end of this lesson, you should be able to:

- Scale time-series features without leaking test information.
- Implement and train an MLP forecaster in PyTorch.
- Evaluate neural forecasts on the original target scale.

## Setup for this lesson

This cell recreates the data and completed prerequisites from earlier lessons, so this notebook can be run in a fresh kernel.

In [ ]:
# Shared forecasting setup from the preceding lesson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.ar_model import AutoReg
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df.columns = ['Date', 'Passengers']
df['Log_Passengers'] = np.log(df['Passengers'])
df['Log_Diff'] = df['Log_Passengers'].diff()
series = df['Log_Diff'].dropna().values

def make_lag_matrix(series, n_lags):
    series = np.asarray(series)
    X, y = [], []
    for t in range(n_lags, len(series)):
        X.append(series[t - n_lags:t])
        y.append(series[t])
    return np.asarray(X), np.asarray(y)

N_LAGS = 12
X, y = make_lag_matrix(series, N_LAGS)
split = int(len(X) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

naive_preds = np.concatenate([[y_train[-1]], y_test[:-1]])
train_series = series[:split + N_LAGS]
ar_result = AutoReg(train_series, lags=N_LAGS).fit()
ar_preds = ar_result.predict(
    start=len(train_series),
    end=len(train_series) + len(y_test) - 1,
    dynamic=False,
)


rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

gb_model = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.1, random_state=42
)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)

def compute_metrics(y_true, y_pred, label='Model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f'{label:<30}  RMSE={rmse:.5f}   MAE={mae:.5f}')
    return {'RMSE': rmse, 'MAE': mae}

results = {
    'Naive Baseline': compute_metrics(y_test, naive_preds, 'Naive Baseline'),
    'AR(12)': compute_metrics(y_test, ar_preds, 'AR(12)'),
    'Random Forest': compute_metrics(y_test, rf_preds, 'Random Forest'),
    'Gradient Boosting': compute_metrics(y_test, gb_preds, 'Gradient Boosting'),
}


***
## Part 5 — MLP for Time Series

### What is an MLP?

A **Multilayer Perceptron (MLP)** is a fully connected feedforward neural network. It maps
the lag feature vector $\mathbf{x} \in \mathbb{R}^p$ to a scalar forecast through stacked
layers of linear transformations and non-linear activations:

$$\mathbf{h}^{(1)} = \sigma(W^{(1)} \mathbf{x} + \mathbf{b}^{(1)})$$
$$\mathbf{h}^{(2)} = \sigma(W^{(2)} \mathbf{h}^{(1)} + \mathbf{b}^{(2)})$$
$$\hat{y} = W^{(3)} \mathbf{h}^{(2)} + b^{(3)}$$

where $\sigma(z) = \max(0, z)$ is the **ReLU** activation function.

**Key advantage:** MLPs can learn non-linear combinations of lag features.

**Key limitation:** The MLP treats the lag vector as an *unordered* set of features —
it doesn't know that `lag_1` is more recent than `lag_12`. RNNs and LSTMs (Part 6)
encode this sequential structure explicitly.

### Why do we need to scale?

Neural networks require **input scaling** because gradient updates are sensitive to feature
magnitude. We use `MinMaxScaler` fitted *only* on the training data:

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}} \in [0, 1]$$

> ⚠️ **Scaling rule:** `fit_transform` on training data only. `transform` on test data.
> Never fit the scaler on test data — this would leak future statistics into training.

### 5.1 Scale the data

In [ ]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# FILL IN: fit_transform on training data, transform on test data
X_train_s = scaler_X.fit_transform(???)   # fit AND transform train
X_test_s  = scaler_X.transform(???)       # transform test ONLY

y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_s  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()      # for reference

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_s,  dtype=torch.float32)

print('X_train_t shape:', X_train_t.shape)
print('y_train_t shape:', y_train_t.shape)

### 5.2 Define the MLP architecture

**Your turn!** Complete the `MLPForecaster` class below. The network should have:
- A linear layer mapping `input_size` → `hidden_size`, followed by ReLU
- A linear layer mapping `hidden_size` → `hidden_size // 2`, followed by ReLU
- An output linear layer mapping `hidden_size // 2` → 1

> 💡 **PyTorch pattern:** Define layers in `__init__` using `nn.Sequential`. Implement
> the forward pass in `forward(self, x)`. `nn.Linear(in, out)` creates a fully connected
> layer. `nn.ReLU()` is the activation.

In [ ]:
class MLPForecaster(nn.Module):
    """
    Simple Multilayer Perceptron for time series forecasting.

    Takes a lag feature vector of length `input_size` and produces
    a single scalar forecast.

    Parameters
    ----------
    input_size  : int, number of lag features
    hidden_size : int, number of neurons in each hidden layer
    """
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, ???),       # FILL IN
            nn.ReLU(),
            nn.Linear(???, hidden_size // 2), # FILL IN
            nn.ReLU(),
            nn.Linear(hidden_size // 2, ???)  # FILL IN: output 1 value
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

### 5.3 Train the MLP

In [ ]:
mlp     = MLPForecaster(input_size=N_LAGS, hidden_size=32)
opt     = torch.optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

EPOCHS = 300
losses = []

for epoch in range(EPOCHS):
    mlp.train()
    opt.zero_grad()
    # FILL IN: forward pass and loss computation
    pred = mlp(???)
    loss = loss_fn(???, y_train_t)
    loss.backward()
    opt.step()
    losses.append(loss.item())

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(losses)
ax.set_title('MLP Training Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (scaled)')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the MLP
mlp.eval()
with torch.no_grad():
    mlp_preds_s = mlp(X_test_t).numpy()    # predictions in scaled space

# FILL IN: inverse-transform predictions back to original (log-diff) scale
mlp_preds = scaler_y.inverse_transform(???).ravel()

results['MLP'] = compute_metrics(y_test, mlp_preds, 'MLP')